Setup

In [ ]:
# Install dependencies and Ollama
import subprocess
import time
import threading
import os

# First install zstd (required for Ollama extraction)
print("Installing zstd...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)

# Install Ollama
print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# Function to run Ollama server
def run_ollama_server():
    subprocess.run(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Start Ollama server in a separate thread (not background process)
print("Starting Ollama server...")
server_thread = threading.Thread(target=run_ollama_server, daemon=True)
server_thread.start()

# Wait for server to initialize
time.sleep(10)

# Pull the model (this will take several minutes)
print("Pulling llama3.1:8b (this takes 5-10 minutes)...")
subprocess.run(["ollama", "pull", "llama3.1:8b"], check=True)
subprocess.run(["ollama", "pull", "qwen2.5:14b "], check=True)

print("Ollama ready!")

In [ ]:
"""
Generates three types of triples, each designed to target a specific
weakness in the training signal:

  Type 1 — Standard   (organizer-style)
    Anchor + Similar (same CoA/Outcomes, different surface) + Distant
    Covers the easy end of the difficulty spectrum.

  Type 2 — Aspect-Controlled
    Precisely vary ONE aspect at a time while holding others constant.
    Four sub-patterns:
      (a) same_coa_diff_outcome  — trains Outcomes heads
      (b) diff_coa_same_outcome  — trains CoA heads
      (c) same_coa_same_outcome  — unambiguous positive (easy)
      (d) diff_coa_diff_outcome  — hard negative with shared theme only

  Type 3 — Hard Negatives
    Both candidates share surface features (genre, setting, character
    types) with the anchor but differ in plot structure.
    Targets the 74 "hard samples" (62.72% system accuracy in the paper)
    where the difficult cases cost most models the most points.

Output format (JSONL):
  {
    "model":           "claude-sonnet-4 | llama3.1:8b | ...",
    "anchor_text":     "...",
    "text_a":          "...",
    "text_b":          "...",
    "text_a_is_closer": true | false,
    "generation_type": "standard | aspect_coa | aspect_outcome | hard_negative",
    "seed_genre":      "...",
    "seed_archetype":  "..."
  }

Usage (Kaggle / Colab with Ollama):
    python build_synthetic_data.py

The script is crash-safe: it writes to JSONL incrementally and resumes
from the last completed triple on restart.
"""

import json
import os
import random
import re
import sys
import time
import urllib.request
from pathlib import Path
from typing import Optional


OUTPUT_PATH      = "/kaggle/working/synthetic_data_new.jsonl"
CACHE_PATH       = "/kaggle/working/synthetic_gen_cache.json"   # crash recovery

BACKEND          = "ollama"  # "ollama" or "api"
OLLAMA_MODEL     = "llama3.1:8b"
OLLAMA_URL       = "http://localhost:11434/api/generate"
API_MODEL        = "claude-sonnet-4-20250514"  

# How many triples to generate per type (total = 1900 - IMPORTANT: Stopped at 1097 triples)
N_STANDARD       = 700
N_ASPECT         = 900   
N_HARD_NEG       = 300
N_TOTAL          = N_STANDARD + N_ASPECT + N_HARD_NEG  

SEED             = 42

# SEED TOPICS

GENRES = [
    "crime thriller", "romantic comedy", "war drama", "science fiction",
    "heist film", "political drama", "coming-of-age", "psychological thriller",
    "adventure film", "supernatural horror", "legal drama", "espionage thriller",
    "road movie", "survival story", "family drama", "sports drama",
    "historical epic", "detective mystery", "redemption story", "revenge thriller",
    "courtroom drama", "dystopian fiction", "western", "medical drama",
    "conspiracy thriller",
]

ARCHETYPES = [
    "a protagonist seeks revenge for a loved one's death",
    "an outsider must prove themselves worthy to join a group",
    "a character discovers a dark secret about their own family",
    "two enemies are forced to cooperate in order to survive",
    "a protagonist must choose between duty and personal loyalty",
    "an underdog competes against a powerful corrupt establishment",
    "a character tries to escape and leave behind their past identity",
    "a mentor and protégé relationship is shattered by betrayal",
    "an investigator slowly realizes they are part of the conspiracy they are uncovering",
    "a protagonist sacrifices everything for the sake of others",
    "a false accusation forces a character to prove their innocence",
    "a character is gradually manipulated and begins to lose touch with reality",
    "strangers from opposing worlds form an unlikely bond during a shared crisis",
    "an obsessive pursuit of a single goal ultimately destroys the pursuer",
    "a character returns home after a long absence to find everything has changed",
    "a person of privilege confronts the consequences of their past actions",
    "two people compete for the same goal but fall in love along the way",
    "a whistleblower exposes a powerful institution at great personal cost",
    "a reformed criminal is pulled back into their old life by circumstances",
    "a character must complete one final mission before they can be free",
]

# PROMPTS

# System message — shared across all prompts
SYSTEM = (
    "You are a creative writing assistant specialising in Wikipedia-style "
    "film plot summaries. You write in a neutral, encyclopedic third-person "
    "tone. You never include titles, years, or any meta-information — only "
    "the plot. Each summary is 5-8 sentences long."
)

# ── Type 1: Standard (organizer-style) ───────────────────────────────────────

SEED_PROMPT = """\
Write a short story of 5-8 sentences in the style of a Wikipedia film plot \
summary. The story should be in the genre of {genre} and feature the \
following central situation: {archetype}.

Do not include a title, year, or meta information. Describe only the plot \
in a neutral, encyclopedic tone, as if summarising the events of a film.
Begin your response immediately with the story. No preamble."""

SIMILAR_PROMPT = """\
Here is a story summary:

{anchor}

Write a new story that closely follows the same general situation and plot \
structure. Keep the same sequence of events and outcome, but change all \
character names, specific locations, and surface details of the setting. \
The result should feel like a clearly different story while mirroring the \
original narrative structure.

Write it in 5-8 sentences, in the style of a Wikipedia film plot summary. \
Do not include a title, year, or meta information — output the plot only. \
Begin immediately with the story."""

DISTANT_PROMPT = """\
Here is a story summary:

{anchor}

Write a new short story of 5-8 sentences that is only loosely inspired by \
the original. You may change the setting, characters, genre, tone, and \
sequence of events. Create a different central conflict and a different \
resolution. The ending should not mirror the source. The story should \
retain only a faint thematic connection or mood from the original, while \
otherwise standing as a distinctly new narrative.

Write it in the style of a Wikipedia film plot summary. Do not include a \
title, year, or meta information — output the plot only. \
Begin immediately with the story."""

# ── Type 2: Aspect-Controlled ─────────────────────────────────────────────────

# (a) Same CoA, different Outcome
SAME_COA_DIFF_OUTCOME = """\
Here is a story summary:

{anchor}

Write a new story that follows EXACTLY the same sequence of plot events \
as the original — the same type of actions occur in the same order, and \
the same causal chain connects them. However, the FINAL OUTCOME must be \
clearly different: the protagonist should end up in a meaningfully \
different state (succeeds vs. fails, survives vs. dies, is reunited vs. \
remains separated, etc.).

Change all character names, specific locations, and surface details. \
Keep the abstract plot structure identical but change only the ending.

Write 5-8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

# (b) Different CoA, same Outcome
DIFF_COA_SAME_OUTCOME = """\
Here is a story summary:

{anchor}

Write a new story where the protagonist ends up in EXACTLY the same final \
state as in the original (the same outcome: achieves the same goal, \
suffers the same loss, reaches the same resolution). However, the PATH \
they take to get there must be completely different — different types of \
events, different obstacles, different causal chain.

Change all character names, specific locations, and surface details. \
Keep only the final outcome the same.

Write 5-8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

# (c) Same CoA AND same Outcome (unambiguous similar)
SAME_COA_SAME_OUTCOME = """\
Here is a story summary:

{anchor}

Write a new story that is narratively very close to the original: the \
same sequence of events occurs in the same order, and the protagonist \
reaches the same final outcome. This should feel like a clear retelling \
of the same story structure.

Change ALL character names, locations, and surface details so it reads \
as a completely different story, but keep the plot structure and ending \
identical.

Write 5-8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

# (d) Different CoA AND different Outcome — hard negative (same theme only)
DIFF_COA_DIFF_OUTCOME = """\
Here is a story summary:

{anchor}

Write a new story that explores the SAME ABSTRACT THEMES and human \
experiences as the original, but has a completely different sequence of \
events AND a different final outcome. The surface situation (genre, \
setting, character types) should feel loosely familiar, but the plot \
structure and ending should be entirely different.

This is a HARD NEGATIVE: the stories should share thematic resonance but \
differ in narrative structure.

Change all names and specific details. Write 5-8 sentences. Wikipedia \
film plot style. No title, year, or meta information. Begin immediately."""

# ── Type 3: Hard Negative ──────────────────────────────────────────────────────

HARD_NEGATIVE_SIMILAR_PROMPT = """\
Here is a story summary:

{anchor}

Write a new story that shares the same genre, the same type of \
protagonist (by role, not name), and the same general setting category \
(e.g., both are war stories, both are courtroom dramas). The new story \
should feel superficially similar on first reading.

However, the PLOT STRUCTURE must be genuinely different: different \
sequence of events, different central conflict, and a clearly different \
outcome. This is a hard negative — similar surface, different narrative \
structure.

Write 5-8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

HARD_NEGATIVE_DISTANT_PROMPT = """\
Here is a story summary:

{anchor}

Write a completely different story in a different genre, with a different \
type of protagonist, different setting, and different plot structure. \
The only connection to the original should be a very faint thematic mood.

Write 5-8 sentences. Wikipedia film plot style. No title, year, or meta \
information. Begin immediately with the story."""

# GENERATION BACKEND

def _ollama_available():
    try:
        urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def _anthropic_available():
    return bool(os.environ.get("ANTHROPIC_API_KEY", ""))


def _generate_ollama(prompt: str, max_tokens: int = 300) -> str:
    payload = json.dumps({
        "model":   OLLAMA_MODEL,
        "prompt":  SYSTEM + "\n\n" + prompt,
        "stream":  False,
        "options": {
            "num_predict":    max_tokens,
            "temperature":    0.8,
            "top_p":          0.95,
            "repeat_penalty": 1.1,
        },
    }).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read()).get("response", "").strip()


def _generate_api(prompt: str, max_tokens: int = 300) -> str:
    """Call the Anthropic /v1/messages API."""
    api_key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not api_key:
        raise RuntimeError("ANTHROPIC_API_KEY not set")

    payload = json.dumps({
        "model":      API_MODEL,
        "max_tokens": max_tokens,
        "system":     SYSTEM,
        "messages":   [{"role": "user", "content": prompt}],
    }).encode()

    req = urllib.request.Request(
        "https://api.anthropic.com/v1/messages",
        data=payload,
        headers={
            "Content-Type":         "application/json",
            "x-api-key":            api_key,
            "anthropic-version":    "2023-06-01",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read())
    return data["content"][0]["text"].strip()


def generate(prompt: str, max_tokens: int = 300) -> str:
    """Generate text with retry logic using configured backend."""
    for attempt in range(3):
        try:
            if BACKEND == "ollama":
                return _generate_ollama(prompt, max_tokens)
            else:
                return _generate_api(prompt, max_tokens)
        except Exception as e:
            if attempt == 2:
                print(f"  [ERROR] Generation failed after 3 attempts: {e}")
                return ""
            wait = 2 ** attempt
            print(f"  [retry {attempt+1}] {e}  — waiting {wait}s")
            time.sleep(wait)
    return ""


def _clean(text: str) -> str:
    """Remove common LLM preambles and leading labels."""
    text = text.strip()
    # "Here is the story: <content>" / "Here's a summary:\n<content>" / "Sure! ..."
    # Matches preamble up to first colon or newline, strips the preamble
    text = re.sub(
        r"^(?:here(?:'s| is)|sure[!,]?|certainly[!,]?)"
        r"(?:[^:\n]*)(?::\s*|\n+)",
        '', text, flags=re.IGNORECASE).strip()
    # **Title:** or **Story:** markdown headers
    text = re.sub(r'^\*\*[^*]+\*\*\s*\n+', '', text)
    # "Story:" or "Summary:" standalone label at start
    text = re.sub(r'^(?:story|summary|plot)\s*:\s*', '', text,
                  flags=re.IGNORECASE)
    return text.strip()


# TRIPLE BUILDERS

def _is_valid(text: str, min_words: int = 20) -> bool:
    return bool(text) and len(text.split()) >= min_words


def build_standard_triple(genre: str, archetype: str,
                           model_name: str) -> Optional[dict]:
    """Type 1: anchor + similar + distant, 50/50 label swap."""
    anchor = _clean(generate(
        SEED_PROMPT.format(genre=genre, archetype=archetype)))
    if not _is_valid(anchor):
        return None

    similar = _clean(generate(SIMILAR_PROMPT.format(anchor=anchor)))
    distant = _clean(generate(DISTANT_PROMPT.format(anchor=anchor)))

    if not _is_valid(similar) or not _is_valid(distant):
        return None

    # 50/50 label swap to prevent positional bias
    if random.random() < 0.5:
        text_a, text_b, closer = similar, distant, True
    else:
        text_a, text_b, closer = distant, similar, False

    return {
        "model":            model_name,
        "anchor_text":      anchor,
        "text_a":           text_a,
        "text_b":           text_b,
        "text_a_is_closer": closer,
        "generation_type":  "standard",
        "seed_genre":       genre,
        "seed_archetype":   archetype,
    }


def build_aspect_triple(genre: str, archetype: str, pattern: str,
                         model_name: str) -> Optional[dict]:
    """
    Type 2: aspect-controlled.
    pattern ∈ {same_coa_diff_outcome, diff_coa_same_outcome,
               same_coa_same_outcome, diff_coa_diff_outcome}
    """
    anchor = _clean(generate(
        SEED_PROMPT.format(genre=genre, archetype=archetype)))
    if not _is_valid(anchor):
        return None

    prompt_map = {
        "same_coa_diff_outcome": SAME_COA_DIFF_OUTCOME,
        "diff_coa_same_outcome": DIFF_COA_SAME_OUTCOME,
        "same_coa_same_outcome": SAME_COA_SAME_OUTCOME,
        "diff_coa_diff_outcome": DIFF_COA_DIFF_OUTCOME,
    }

    # For d: diff_coa_diff_outcome → the aspect-variant IS the distant story
    # and we need a clearly similar story as the other candidate
    if pattern in ("same_coa_diff_outcome", "diff_coa_same_outcome",
                   "same_coa_same_outcome"):
        # aspect_variant is the SIMILAR story
        aspect_variant = _clean(
            generate(prompt_map[pattern].format(anchor=anchor)))
        distant = _clean(generate(DISTANT_PROMPT.format(anchor=anchor)))

        if not _is_valid(aspect_variant) or not _is_valid(distant):
            return None

        if random.random() < 0.5:
            text_a, text_b, closer = aspect_variant, distant, True
        else:
            text_a, text_b, closer = distant, aspect_variant, False

    else:  # diff_coa_diff_outcome → hard negative case
        hard_neg = _clean(
            generate(prompt_map[pattern].format(anchor=anchor)))
        similar = _clean(generate(SIMILAR_PROMPT.format(anchor=anchor)))

        if not _is_valid(hard_neg) or not _is_valid(similar):
            return None

        if random.random() < 0.5:
            text_a, text_b, closer = similar, hard_neg, True
        else:
            text_a, text_b, closer = hard_neg, similar, False

    return {
        "model":            model_name,
        "anchor_text":      anchor,
        "text_a":           text_a,
        "text_b":           text_b,
        "text_a_is_closer": closer,
        "generation_type":  f"aspect_{pattern}",
        "seed_genre":       genre,
        "seed_archetype":   archetype,
    }


def build_hard_negative_triple(genre: str, archetype: str,
                                model_name: str) -> Optional[dict]:
    """
    Type 3: both candidates are surface-similar to anchor
    but one has a genuinely different plot structure.
    """
    anchor = _clean(generate(
        SEED_PROMPT.format(genre=genre, archetype=archetype)))
    if not _is_valid(anchor):
        return None

    # hard_similar: same genre/setting/character type but different plot
    hard_similar = _clean(
        generate(HARD_NEGATIVE_SIMILAR_PROMPT.format(anchor=anchor)))
    # hard_distant: completely different
    hard_distant = _clean(
        generate(HARD_NEGATIVE_DISTANT_PROMPT.format(anchor=anchor)))

    if not _is_valid(hard_similar) or not _is_valid(hard_distant):
        return None

    # hard_similar is NOT closer to anchor (different plot structure)
    # hard_distant is even further
    # → Neither is "clearly" closer; this is genuinely ambiguous
    # We label hard_distant as "not closer" since it's more clearly different
    # → text_a = hard_distant, text_b = hard_similar, text_a_is_closer = False
    if random.random() < 0.5:
        text_a, text_b, closer = hard_distant, hard_similar, False
    else:
        text_a, text_b, closer = hard_similar, hard_distant, True

    return {
        "model":            model_name,
        "anchor_text":      anchor,
        "text_a":           text_a,
        "text_b":           text_b,
        "text_a_is_closer": closer,
        "generation_type":  "hard_negative",
        "seed_genre":       genre,
        "seed_archetype":   archetype,
    }


# CRASH-SAFE CACHE

def load_cache(path: str) -> set:
    """Return set of already-generated (genre, archetype, type) keys."""
    if not Path(path).exists():
        return set()
    done = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                done.add((obj["seed_genre"], obj["seed_archetype"],
                           obj["generation_type"]))
            except Exception:
                pass
    return done


def append_result(path: str, result: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")


# GENERATION PLAN BUILDER

def build_plan(rng: random.Random) -> list:
    """
    Build a randomised list of (genre, archetype, triple_type) tasks.
    Balanced across types, genres, archetypes.
    """
    aspect_patterns = [
        "same_coa_diff_outcome", "diff_coa_same_outcome",
        "same_coa_same_outcome", "diff_coa_diff_outcome",
    ]
    n_per_aspect = N_ASPECT // len(aspect_patterns)

    tasks = []

    # Standard
    for _ in range(N_STANDARD):
        tasks.append({
            "genre":     rng.choice(GENRES),
            "archetype": rng.choice(ARCHETYPES),
            "type":      "standard",
        })

    # Aspect-controlled
    for pat in aspect_patterns:
        for _ in range(n_per_aspect):
            tasks.append({
                "genre":     rng.choice(GENRES),
                "archetype": rng.choice(ARCHETYPES),
                "type":      f"aspect_{pat}",
            })

    # Hard negatives
    for _ in range(N_HARD_NEG):
        tasks.append({
            "genre":     rng.choice(GENRES),
            "archetype": rng.choice(ARCHETYPES),
            "type":      "hard_negative",
        })

    rng.shuffle(tasks)
    return tasks


# MAIN

def main():
    # Set model name based on backend
    if BACKEND == "ollama":
        model_tag = OLLAMA_MODEL
        print(f"Backend: Ollama ({OLLAMA_MODEL})")
        
        # Check if Ollama is available
        if not _ollama_available():
            print("ERROR: Ollama is not available. Please ensure Ollama is running.")
            print("Run: ollama serve && ollama pull llama3.1:8b")
            sys.exit(1)
    else:
        model_tag = API_MODEL
        print(f"Backend: Anthropic API ({API_MODEL})")
        
        # Check if API key is available
        if not _anthropic_available():
            print("ERROR: ANTHROPIC_API_KEY not set.")
            sys.exit(1)

    print(f"Output: {OUTPUT_PATH}")
    print(f"Target: {N_STANDARD} standard + "
          f"{N_ASPECT} aspect-controlled + "
          f"{N_HARD_NEG} hard negatives = "
          f"{N_STANDARD + N_ASPECT + N_HARD_NEG} total")
    print()

    rng = random.Random(SEED)
    plan = build_plan(rng)

    # Load crash-recovery cache (the output file acts as its own cache)
    done_keys = load_cache(OUTPUT_PATH)
    print(f"Already generated: {len(done_keys)} triples (resuming)")

    # Stats tracking
    counts   = {"standard": 0, "hard_negative": 0}
    for p in ["same_coa_diff_outcome", "diff_coa_same_outcome",
               "same_coa_same_outcome", "diff_coa_diff_outcome"]:
        counts[f"aspect_{p}"] = 0
    errors   = 0
    start    = time.time()

    for i, task in enumerate(plan):
        triple_type = task["type"]
        genre       = task["genre"]
        archetype   = task["archetype"]

        cache_key = (genre, archetype, triple_type)
        if cache_key in done_keys:
            continue

        try:
            if triple_type == "standard":
                result = build_standard_triple(
                    genre, archetype, model_tag)
            elif triple_type.startswith("aspect_"):
                pattern = triple_type[len("aspect_"):]
                result  = build_aspect_triple(
                    genre, archetype, pattern, model_tag)
            else:  # hard_negative
                result  = build_hard_negative_triple(
                    genre, archetype, model_tag)

            if result is None:
                errors += 1
                print(f"  [skip] {triple_type} — generation too short")
                continue

            append_result(OUTPUT_PATH, result)
            done_keys.add(cache_key)
            counts[triple_type] = counts.get(triple_type, 0) + 1

        except Exception as e:
            errors += 1
            print(f"  [ERROR] {triple_type} ({genre}/{archetype}): {e}")
            continue

        total_done = sum(counts.values())
        if total_done % 50 == 0 and total_done > 0:
            elapsed   = time.time() - start
            rate      = total_done / elapsed
            remaining = (len(plan) - i) / rate / 60
            print(f"  [{total_done}/{len(plan)}] "
                  f"{rate:.2f} triples/s  ETA {remaining:.1f} min  "
                  f"errors={errors}")
            print(f"  Counts: {counts}")

    # Final summary
    elapsed   = time.time() - start
    total     = sum(counts.values())
    print()
    print("=" * 55)
    print(f"Generation complete in {elapsed/60:.1f} min")
    print(f"Total triples written: {total}")
    print(f"Errors / skipped    : {errors}")
    print(f"Breakdown: {counts}")
    print(f"Output file: {OUTPUT_PATH}")

    # Validate balance
    all_results = []
    with open(OUTPUT_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                all_results.append(json.loads(line))
            except Exception:
                pass

    n_closer = sum(1 for r in all_results if r["text_a_is_closer"])
    n_total  = len(all_results)
    print(f"\nLabel balance: text_a_is_closer=True: "
          f"{n_closer}/{n_total} ({n_closer/n_total*100:.1f}%)")
    if not 0.40 <= n_closer / n_total <= 0.60:
        print("WARNING: Label imbalance detected. Check positional bias in prompts.")
    else:
        print("Label balance OK")
    print("=" * 55)


if __name__ == "__main__":
    main()